In [2]:
# =========================================
# CREDIT RISK - CLEAN PIPELINE
# =========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix


# =========================================
# 1. LOAD DATA
# =========================================
url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
df = pd.read_csv(url)

print("✅ Data loaded")


# =========================================
# 2. FIX TARGET (CORRECT WAY)
# =========================================
# Original dataset: 1 = good, 0 = bad
# We want: 0 = good, 1 = bad

df["credit_risk"] = df["credit_risk"].map({1: 0, 0: 1})

print("\nTarget distribution:")
print(df["credit_risk"].value_counts())


# =========================================
# 3. SPLIT DATA
# =========================================
X = df.drop("credit_risk", axis=1)
y = df["credit_risk"]

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


# =========================================
# 4. PREPROCESSING
# =========================================
numeric_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


# =========================================
# 5. MODELS
# =========================================
models = {
    "Logistic": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
    "SVM": SVC(probability=True)
}


# =========================================
# 6. CROSS VALIDATION
# =========================================
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=42)

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        Xtrain,
        ytrain,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    print(f"\n🚀 {name}")
    print("------------------------")

    row = {"Model": name}

    for metric in scoring:
        val_score = np.mean(scores[f"test_{metric}"])
        print(f"{metric}: {val_score:.3f}")
        row[metric] = val_score

    results.append(row)


# =========================================
# 7. RESULTS TABLE
# =========================================
df_results = pd.DataFrame(results).sort_values(by="roc_auc", ascending=False)

print("\n📊 MODEL COMPARISON")
print(df_results)


# =========================================
# 8. FINAL MODEL (BEST ONE)
# =========================================
best_model_name = df_results.iloc[0]["Model"]
print(f"\n🏆 Best Model: {best_model_name}")

best_model = models[best_model_name]

final_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(Xtrain, ytrain)

y_pred = final_pipeline.predict(Xtest)


# =========================================
# 9. FINAL EVALUATION
# =========================================
print("\n📊 FINAL TEST PERFORMANCE")
print(classification_report(ytest, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, y_pred))

✅ Data loaded

Target distribution:
credit_risk
0    700
1    300
Name: count, dtype: int64

🚀 Logistic
------------------------
accuracy: 0.696
precision: 0.495
recall: 0.658
f1: 0.565
roc_auc: 0.762

🚀 RandomForest
------------------------
accuracy: 0.745
precision: 0.677
recall: 0.296
f1: 0.409
roc_auc: 0.782

🚀 SVM
------------------------
accuracy: 0.748
precision: 0.678
recall: 0.319
f1: 0.431
roc_auc: 0.782

📊 MODEL COMPARISON
          Model  accuracy  precision    recall        f1   roc_auc
2           SVM  0.748130   0.678272  0.318750  0.431021  0.782244
1  RandomForest  0.745006   0.677061  0.295833  0.409324  0.782059
0      Logistic  0.695625   0.495309  0.658333  0.564538  0.761524

🏆 Best Model: SVM

📊 FINAL TEST PERFORMANCE
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       140
           1       0.72      0.48      0.58        60

    accuracy                           0.79       200
   macro avg       0.77      0.